# Master Workflow: EGFR Graph Neural Networks & Generative Models

This notebook orchestrates the complete pipeline for:
1. **Data Preparation**: Download ZINC 250k and process EGFR bioactivity data
2. **Model Architectures**: Define GraphVAE and EGFR GNN models
3. **Pocket Conditioning**: Extract T790M pocket embeddings
4. **Training**: Two-phase GraphVAE training (ZINC pre-training → EGFR fine-tuning) and EGFR GNN training
5. **Evaluation**: Evaluate trained models on test sets

## Workflow Overview

```
ZINC 250k Dataset
       ↓
Data Preparation (SMILES → PyG Data)
       ↓
  ┌────┴────┐
  ↓         ↓
GraphVAE  EGFR GNN
  ↓         ↓
Phase 1: Pretrain on ZINC → Phase 2: Finetune on EGFR
  ↓         ↓
Evaluate & Generate Molecules
```

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Add src to path for imports
src_path = Path('.').resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

logger.info(f"Project path: {src_path}")
logger.info("Imports configured successfully")

## 2. Configuration

In [ ]:
# ============================================================================
# WORKFLOW CONFIGURATION
# ============================================================================

config = {
    # Paths
    'project_root': '.',
    'data_dir': 'data',
    'raw_data_dir': 'data/raw',
    'processed_data_dir': 'data/processed',
    'models_dir': 'models',
    'logs_dir': 'logs',
    
    # Dataset files
    'zinc_file': 'data/raw/zinc250k.csv',
    'egfr_graph_file': 'data/processed/graph_data.pt',
    'test_file': 'data/raw/test_smiles_small.csv',
    
    # Model files
    'zinc_pretrained': 'models/zinc_pretrained_graphvae.pth',
    'best_vae': 'models/best_graphvae.pth',
    'best_gnn': 'models/best_gnn_regressor.pth',
    
    # Training hyperparameters - ZINC Pre-training
    'zinc_samples': 50000,
    'pretrain_epochs': 25,
    'pretrain_batch_size': 16,
    'pretrain_lr': 1e-3,
    
    # Training hyperparameters - EGFR Fine-tuning
    'egfr_samples': None,  # Use all
    'finetune_epochs': 50,
    'finetune_batch_size': 16,
    'finetune_lr': 1e-3,
    
    # Model architecture
    'latent_dim': 64,
    'hidden_dim': 128,
    'max_nodes': 50,
    'dropout': 0.2,
    'pocket_dim': 128,
    
    # Pocket extraction
    'pdb_file': None,  # Optional custom PDB
    'ligand_code': 'W2R',
    'pocket_radius': 7.0,
    
    # Device and random seed
    'device': 'cpu',  # or 'cuda' if available
    'seed': 42,
}

# Validate device
import torch
if config['device'] == 'cuda' and not torch.cuda.is_available():
    logger.warning("CUDA requested but not available. Switching to CPU.")
    config['device'] = 'cpu'

logger.info(f"Config loaded - Device: {config['device']}")
logger.info(f"Pretrain: {config['pretrain_epochs']} epochs, Finetune: {config['finetune_epochs']} epochs")

## 3. Download & Prepare Data

In [ ]:
# ============================================================================
# STEP 1: Download ZINC 250k Dataset
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 1: Download ZINC 250k Dataset")
logger.info("="*80)

from download_zinc import download_zinc_250k

zinc_path = download_zinc_250k(
    output_dir=config['raw_data_dir'],
    filename='zinc250k.csv'
)

if zinc_path:
    logger.info(f"✓ ZINC dataset ready: {zinc_path}")
else:
    logger.warning("✗ Failed to download ZINC dataset")

In [ ]:
# ============================================================================
# STEP 2: Data Preparation (SMILES → PyG Data)
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 2: Data Preparation")
logger.info("="*80)

from data_prep import process, smiles_to_pyg_data
import pandas as pd

# Check if EGFR data exists
if Path(config['egfr_graph_file']).exists():
    logger.info(f"✓ EGFR graph data exists: {config['egfr_graph_file']}")
else:
    logger.warning(f"✗ EGFR graph data not found at {config['egfr_graph_file']}")
    logger.info("  Run data_prep.ipynb to generate it from raw EGFR CSV files.")

logger.info("Data preparation utilities loaded")

## 4. Model Architectures

In [ ]:
# ============================================================================
# STEP 3: Load Model Architectures
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 3: Load Model Architectures")
logger.info("="*80)

from generative_model import GraphVAE
from model_arch import EGFR_GNN_Regressor

# Initialize GraphVAE
vae_model = GraphVAE(
    latent_dim=config['latent_dim'],
    hidden_dim=config['hidden_dim'],
    max_nodes=config['max_nodes'],
    dropout=config['dropout'],
    use_pocket_conditioning=True,
).to(config['device'])

logger.info(f"GraphVAE parameters: {sum(p.numel() for p in vae_model.parameters()):,}")

# Initialize EGFR GNN
pred_model = EGFR_GNN_Regressor().to(config['device'])
logger.info(f"EGFR GNN parameters: {sum(p.numel() for p in pred_model.parameters()):,}")

logger.info("✓ Models loaded successfully")

## 5. Pocket Conditioning

In [ ]:
# ============================================================================
# STEP 4: Extract T790M Pocket Embedding
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 4: Extract T790M Pocket Embedding")
logger.info("="*80)

from pocket_extraction import get_pocket_embedding
import torch

try:
    pocket_embedding = get_pocket_embedding(
        pdb_path='data/raw/pdb/3W2S.pdb',
        ligand_code=config['ligand_code'],
        pocket_radius=config['pocket_radius'],
        embedding_dim=config['pocket_dim']
    ).to(config['device'])
    logger.info(f"✓ Pocket embedding loaded: shape {tuple(pocket_embedding.shape)}")
except Exception as e:
    logger.warning(f"Could not load pocket embedding: {e}")
    logger.info("  Using zero embedding for fine-tuning")
    pocket_embedding = torch.zeros(config['pocket_dim'], device=config['device'])

## 6. Training - GraphVAE (Two-Phase)

In [ ]:
# ============================================================================
# STEP 5: Train GraphVAE (Two-Phase)
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 5: GraphVAE Two-Phase Training")
logger.info("="*80)
logger.info("Phase 1: Pre-train on ZINC 250k")
logger.info("Phase 2: Fine-tune on EGFR + T790M pocket conditioning")
logger.info("="*80)

# Import training utilities from train_vae notebook
# Note: For actual training, run train_vae.ipynb directly
# This is a placeholder showing the training flow

logger.info("""
To run GraphVAE training:
  1. Open train_vae.ipynb
  2. Update config parameters as needed
  3. Run all cells
  
Training configuration:
  - ZINC pre-training: %d epochs, batch size %d, lr %.1e
  - EGFR fine-tuning: %d epochs, batch size %d, lr %.1e
  - Latent dimension: %d
  - Max nodes: %d
""" % (
    config['pretrain_epochs'],
    config['pretrain_batch_size'],
    config['pretrain_lr'],
    config['finetune_epochs'],
    config['finetune_batch_size'],
    config['finetune_lr'],
    config['latent_dim'],
    config['max_nodes']
))

## 7. Training - EGFR GNN

In [ ]:
# ============================================================================
# STEP 6: Train EGFR GNN Regressor
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 6: EGFR GNN Regression Training")
logger.info("="*80)

# Import training utilities from train notebook
from train import train_epoch
from torch_geometric.loader import DataLoader
import torch.optim as optim

logger.info("""
To run EGFR GNN training:
  1. Open train.ipynb
  2. Load EGFR graph data from data/processed/graph_data.pt
  3. Run all cells
  
Training configuration:
  - Batch size: 32
  - Learning rate: 1e-3
  - Optimizer: Adam
  - Loss: MSE on pChEMBL values
""")

## 8. Evaluation & Results

In [ ]:
# ============================================================================
# STEP 7: Model Evaluation
# ============================================================================

logger.info("\n" + "="*80)
logger.info("STEP 7: Model Evaluation")
logger.info("="*80)

from evaluate import predict_batch, predict_single

# Check for trained models
models_found = {
    'VAE (ZINC pretrained)': Path(config['zinc_pretrained']).exists(),
    'VAE (Best fine-tuned)': Path(config['best_vae']).exists(),
    'GNN (Best)': Path(config['best_gnn']).exists(),
}

logger.info("Trained models status:")
for model_name, found in models_found.items():
    status = "✓ Found" if found else "✗ Not found"
    logger.info(f"  {status}: {model_name}")

logger.info("""
To evaluate models:
  1. Open evaluate.ipynb
  2. Load trained model checkpoints
  3. Run inference on test set
  4. Compute metrics (MSE, MAE for GNN; reconstruction quality for VAE)
""")

## 9. Summary & Next Steps

In [ ]:
logger.info("\n" + "="*80)
logger.info("WORKFLOW SUMMARY")
logger.info("="*80)

summary = f"""
Project Structure:
  📁 data/
    ├── raw/              (ZINC 250k, EGFR bioactivity, PDB files)
    └── processed/        (PyG graph data objects)
  📁 models/              (Trained checkpoints)
  📁 logs/                (TensorBoard logs)
  📁 src/
    ├── data_prep.ipynb
    ├── download_zinc.ipynb
    ├── generative_model.ipynb
    ├── model_arch.ipynb
    ├── pocket_extraction.ipynb
    ├── evaluate.ipynb
    ├── train.ipynb
    ├── train_vae.ipynb
    └── 00-master-workflow.ipynb (this file)

Notebook Execution Order:
  1. Run data_prep.ipynb (if needed) → generates EGFR graph data
  2. Run train_vae.ipynb → trains GraphVAE model
  3. Run train.ipynb → trains EGFR GNN
  4. Run evaluate.ipynb → evaluate both models

Key Parameters (configured above):
  - Latent dim: {config['latent_dim']}
  - Hidden dim: {config['hidden_dim']}
  - Max nodes: {config['max_nodes']}
  - Pretrain epochs: {config['pretrain_epochs']}
  - Finetune epochs: {config['finetune_epochs']}
  - Device: {config['device']}

Next Steps:
  1. ✓ Review configuration above
  2. → Run train_vae.ipynb for GraphVAE training
  3. → Run train.ipynb for EGFR GNN training
  4. → Run evaluate.ipynb to assess results
"""

logger.info(summary)
logger.info("="*80)